# Chess Win Predictor

This notebook aims to predict the outcome of a chess game (White wins, Black wins, or Draw) based on various features such as player ratings, time control, and opening moves.

## 1. Data Loading
First, we download the dataset from Kaggle and locate the CSV file.

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("arevel/chess-games")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'chess-games' dataset.
Path to dataset files: /kaggle/input/chess-games


In [4]:
!cd /root/.cache/kagglehub/datasets/arevel/chess-games/versions/1 && pwd && ls

/root/.cache/kagglehub/datasets/arevel/chess-games/versions/1
chess_games.csv


In [5]:
import os 

csv_file = os.path.join(path, "chess_games.csv")

## 2. Data Exploration
Load the dataset into a pandas DataFrame and inspect its structure, columns, and data types.

In [6]:
import pandas as pd

df_raw = pd.read_csv(csv_file)
# df = df.sample(n=100000, random_state=42)
df_raw.head()

,Event,White,Black,Result,UTCDate,UTCTime,WhiteElo,BlackElo,WhiteRatingDiff,BlackRatingDiff,ECO,Opening,TimeControl,Termination,AN
0,Classical,eisaaaa,HAMID449,1-0,2016.06.30,22:00:01,1901,1896,11.0,-11.0,D10,Slav Defense,300+5,Time forfeit,1. d4 d5 2. c4 c6 3. e3 a6 4. Nf3 e5 5. cxd5 e...
1,Blitz,go4jas,Sergei1973,0-1,2016.06.30,22:00:01,1641,1627,-11.0,12.0,C20,King's Pawn Opening: 2.b3,300+0,Normal,1. e4 e5 2. b3 Nf6 3. Bb2 Nc6 4. Nf3 d6 5. d3 ...
2,Blitz tournament,Evangelistaizac,kafune,1-0,2016.06.30,22:00:02,1647,1688,13.0,-13.0,B01,Scandinavian Defense: Mieses-Kotroc Variation,180+0,Time forfeit,1. e4 d5 2. exd5 Qxd5 3. Nf3 Bg4 4. Be2 Nf6 5....
3,Correspondence,Jvayne,Wsjvayne,1-0,2016.06.30,22:00:02,1706,1317,27.0,-25.0,A00,Van't Kruijs Opening,-,Normal,1. e3 Nf6 2. Bc4 d6 3. e4 e6 4. Nf3 Nxe4 5. Nd...
4,Blitz tournament,kyoday,BrettDale,0-1,2016.06.30,22:00:02,1945,1900,-14.0,13.0,B90,"Sicilian Defense: Najdorf, Lipnitsky Attack",180+0,Time forfeit,1. e4 c5 2. Nf3 d6 3. d4 cxd4 4. Nxd4 Nf6 5. N...


In [7]:
df_raw.columns

Index(['Event', 'White', 'Black', 'Result', 'UTCDate', 'UTCTime', 'WhiteElo',
       'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff', 'ECO', 'Opening',
       'TimeControl', 'Termination', 'AN'],
      dtype='object')

In [8]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6256184 entries, 0 to 6256183
Data columns (total 15 columns):
 #   Column           Dtype  
---  ------           -----  
 0   Event            object 
 1   White            object 
 2   Black            object 
 3   Result           object 
 4   UTCDate          object 
 5   UTCTime          object 
 6   WhiteElo         int64  
 7   BlackElo         int64  
 8   WhiteRatingDiff  float64
 9   BlackRatingDiff  float64
 10  ECO              object 
 11  Opening          object 
 12  TimeControl      object 
 13  Termination      object 
 14  AN               object 
dtypes: float64(2), int64(2), object(11)
memory usage: 716.0+ MB


## 3. Feature Engineering
Create a new feature `EloDifference` which represents the difference in Elo ratings between the White and Black players. This could be a strong predictor of the game outcome.

In [9]:
df_raw['EloDifference'] = df_raw['WhiteElo'] - df_raw['BlackElo']


## 4. Data Cleaning
Drop columns that are not useful for prediction or contain redundant information. Also, filter out rows with invalid or missing values in critical columns like `TimeControl` and `Result`.

In [10]:
columns_to_drop = ['Event','White','Black','UTCDate','UTCTime','Opening','Termination','AN','WhiteRatingDiff','BlackRatingDiff']

df_cleaned = df_raw.drop(columns=columns_to_drop)
df_cleaned.head()

,Result,WhiteElo,BlackElo,ECO,TimeControl,EloDifference
0,1-0,1901,1896,D10,300+5,5
1,0-1,1641,1627,C20,300+0,14
2,1-0,1647,1688,B01,180+0,-41
3,1-0,1706,1317,A00,-,389
4,0-1,1945,1900,B90,180+0,45


In [11]:
df_cleaned = df_cleaned[df_cleaned['TimeControl'] != '-']
df_cleaned = df_cleaned[df_cleaned['Result'] != '*']

In [12]:
df_cleaned.head()

,Result,WhiteElo,BlackElo,ECO,TimeControl,EloDifference
0,1-0,1901,1896,D10,300+5,5
1,0-1,1641,1627,C20,300+0,14
2,1-0,1647,1688,B01,180+0,-41
4,0-1,1945,1900,B90,180+0,45
5,0-1,1773,1809,C27,180+0,-36


## 5. Data Preprocessing
Encode categorical variables like `Result`, `ECO` (Encyclopedia of Chess Openings), and `TimeControl` into numerical values using `LabelEncoder` so they can be used by machine learning models. Drop any remaining missing values.

In [13]:
from sklearn.preprocessing import LabelEncoder

le_result = LabelEncoder()
df_cleaned['Result'] = le_result.fit_transform(df_cleaned['Result'])

le_eco = LabelEncoder()
df_cleaned['ECO'] = le_eco.fit_transform(df_cleaned['ECO'])

le_time = LabelEncoder()
df_cleaned['TimeControl'] = le_time.fit_transform(df_cleaned['TimeControl'])

df_cleaned = df_cleaned.dropna()

print(df_cleaned.head())



   Result  WhiteElo  BlackElo  ECO  TimeControl  EloDifference
0       1      1901      1896  310          408              5
1       0      1641      1627  220          384             14
2       1      1647      1688  101          203            -41
4       0      1945      1900  190          203             45
5       0      1773      1809  227          203            -36


In [14]:
# check lables for Result column

print("Unique labels in 'Result' column:", df_cleaned['Result'].unique())

Unique labels in 'Result' column: [1 0 2]


## 6. Handling Class Imbalance
The dataset might have an unequal number of wins for White, Black, and Draws. We balance the dataset by downsampling the majority classes to match the number of samples in the minority class. This prevents the model from being biased towards the majority class.

In [15]:
from sklearn.utils import resample

df_black = df_cleaned[df_cleaned['Result'] == 0]
df_white = df_cleaned[df_cleaned['Result'] == 1]
df_draw = df_cleaned[df_cleaned['Result'] == 2]

min_samples = min(len(df_black), len(df_white), len(df_draw))
print(f"Minimum samples per class: {min_samples}")

Minimum samples per class: 237589


In [16]:
df_black_balanced = resample(df_black, replace=False, n_samples=min_samples, random_state=42)
df_white_balanced = resample(df_white, replace=False, n_samples=min_samples, random_state=42)
df_draw_balanced = resample(df_draw, replace=False, n_samples=min_samples, random_state=42)

In [17]:
df_balanced = pd.concat([df_black_balanced, df_white_balanced, df_draw_balanced])

In [18]:
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

In [19]:
print(df_balanced['Result'].value_counts())

Result
2    237589
0    237589
1    237589
Name: count, dtype: int64


## 7. Train-Test Split
Separate the features (`X`) from the target variable (`y`). Then, split the dataset into training and testing sets to evaluate the model's performance on unseen data.

In [20]:
X = df_balanced.drop(columns=['Result'])
y = df_balanced['Result']

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

Training set shape: (570213, 5)
Test set shape: (142554, 5)


## 8. Feature Scaling
Standardize the features by removing the mean and scaling to unit variance. This is particularly important for models like Logistic Regression that are sensitive to the scale of the input features.

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 9. Model Training
Train two different classification models:
1. **Random Forest Classifier**: An ensemble learning method that operates by constructing a multitude of decision trees.
2. **Logistic Regression**: A linear model for classification.

In [23]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_scaled, y_train)

RandomForestClassifier(random_state=42)

In [25]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)


LogisticRegression(max_iter=1000, random_state=42)

## 10. Model Evaluation
Evaluate the performance of both models on the test set using metrics like precision, recall, and F1-score. This helps us understand how well the models generalize to unseen data and which model performs better for this specific task.

In [26]:
from sklearn.metrics import classification_report, confusion_matrix


rf_pred = rf_model.predict(X_test_scaled)   
lr_pred = lr_model.predict(X_test_scaled)




In [28]:
print("="*40)
print("   LOGISTIC REGRESSION     ")
print("="*40)
print(classification_report(y_test, lr_pred, target_names=['Black Wins (0)', 'White Wins (1)', 'Draw (2)']))

print("\n" + "="*40)
print("     RANDOM FOREST       ")
print("="*40)
print(classification_report(y_test, rf_pred, target_names=['Black Wins (0)', 'White Wins (1)', 'Draw (2)']))

   LOGISTIC REGRESSION     
                precision    recall  f1-score   support

Black Wins (0)       0.47      0.52      0.49     47414
White Wins (1)       0.47      0.53      0.50     47527
      Draw (2)       0.41      0.32      0.36     47613

      accuracy                           0.46    142554
     macro avg       0.45      0.46      0.45    142554
  weighted avg       0.45      0.46      0.45    142554


     RANDOM FOREST       
                precision    recall  f1-score   support

Black Wins (0)       0.44      0.46      0.45     47414
White Wins (1)       0.44      0.46      0.45     47527
      Draw (2)       0.38      0.36      0.37     47613

      accuracy                           0.42    142554
     macro avg       0.42      0.42      0.42    142554
  weighted avg       0.42      0.42      0.42    142554



In [30]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

print("Training Tuned Random Forest model...")

# New model with Hyperparameter Tuning
rf_model_tuned = RandomForestClassifier(
    n_estimators=300,        # Increase number of trees from 100 to 300
    max_depth=15,            # Limit maximum depth of the tree to 15
    min_samples_split=20,    # Minimum number of samples required to split an internal node
    min_samples_leaf=10,     # Minimum number of samples required to be at a leaf node
    random_state=42,
    n_jobs=-1                # Use all processors to speed up training
)

rf_model_tuned.fit(X_train_scaled, y_train)
y_pred_tuned = rf_model_tuned.predict(X_test_scaled)

print("\n" + "="*45)
print(" TUNED RANDOM FOREST RESULTS ")
print("="*45)
print(classification_report(y_test, y_pred_tuned, target_names=['Black Wins', 'White Wins', 'Draw']))

Tuned Random Forest ආකෘතිය පුහුණු වෙමින් පවතී...

 TUNED RANDOM FOREST ප්‍රතිඵල 


NameError: name 'y_test_n' is not defined

In [ ]:
import joblib

# Save the tuned model, scaler, and label encoders to files
joblib.dump(rf_model_tuned, 'chess_rf_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(le_eco, 'le_eco.pkl')
joblib.dump(le_time, 'le_time.pkl')

print("Model and preprocessors saved successfully!")